In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 01 — Data Collection\n",
    "## Football Defence Analysis\n",
    "\n",
    "**Stage 2: Data Acquisition and Understanding**\n",
    "\n",
    "This notebook fetches standings data for all 5 leagues x 5 seasons.\n",
    "All responses are cached to data/raw/ so reruns cost 0 API requests.\n",
    "\n",
    "### Before running:\n",
    "1. Ensure .env file exists with your API key\n",
    "2. Run: pip install -r requirements.txt\n",
    "3. Activate your virtual environment"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "sys.path.insert(0, '..')\n",
    "\n",
    "import pandas as pd\n",
    "from IPython.display import display\n",
    "\n",
    "from config import LEAGUES, SEASONS, DATA_PROC_DIR\n",
    "from src.collectors.api_client import APIFootballClient\n",
    "from src.collectors.data_fetcher import fetch_standings, extract_top4\n",
    "\n",
    "print('Imports successful')\n",
    "print(f'Leagues configured: {list(LEAGUES.keys())}')\n",
    "print(f'Seasons configured: {SEASONS}')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ============================================================\n",
    "# METADATA REVIEW\n",
    "# Document what each field means before fetching\n",
    "# ============================================================\n",
    "metadata = {\n",
    "    'position':          'Final league standing (1 = champion)',\n",
    "    'points':            'Total season points (W=3, D=1, L=0)',\n",
    "    'goals_against':     'Total goals conceded (PRIMARY metric)',\n",
    "    'goals_for':         'Total goals scored',\n",
    "    'games_played':      'Matches played (34 or 38 per league)',\n",
    "    'goals_against_avg': 'Goals conceded per match (derived)',\n",
    "    'goal_diff':         'GF minus GA',\n",
    "}\n",
    "\n",
    "print('Field definitions:')\n",
    "for field, desc in metadata.items():\n",
    "    print(f'  {field:25s}: {desc}')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ============================================================\n",
    "# INITIAL TRIAGE\n",
    "# Estimate request count before fetching\n",
    "# Free plan = 100 requests/day\n",
    "# ============================================================\n",
    "n_leagues  = len(LEAGUES)\n",
    "n_seasons  = len(SEASONS)\n",
    "n_requests = n_leagues * n_seasons\n",
    "\n",
    "print(f'Estimated requests: {n_leagues} leagues x {n_seasons} seasons = {n_requests}')\n",
    "print(f'Free plan limit: 100/day')\n",
    "print(f'Within budget: {\"YES\" if n_requests <= 100 else \"NO\"}')\n",
    "print(f'Note: Cached responses cost 0 requests on reruns')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ============================================================\n",
    "# FETCH STANDINGS DATA\n",
    "# First run: up to 25 API requests\n",
    "# Subsequent runs: 0 requests (served from cache)\n",
    "# ============================================================\n",
    "client = APIFootballClient()\n",
    "df_raw = fetch_standings(client)\n",
    "\n",
    "print(f'Fetched {len(df_raw)} rows')\n",
    "print(f'Live API requests this session: {client.live_request_count}')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ============================================================\n",
    "# DATA OVERVIEW\n",
    "# ============================================================\n",
    "print('Shape:', df_raw.shape)\n",
    "print('\\nColumn dtypes:')\n",
    "print(df_raw.dtypes)\n",
    "print('\\nFirst 5 rows:')\n",
    "display(df_raw.head())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ============================================================\n",
    "# COVERAGE CHECK\n",
    "# Did we get all 25 league-seasons?\n",
    "# ============================================================\n",
    "coverage = df_raw.groupby(['league_key', 'season']).size().unstack(fill_value=0)\n",
    "print('Teams per league-season (0 = missing):')\n",
    "display(coverage)\n",
    "\n",
    "missing = coverage[coverage == 0].stack()\n",
    "if len(missing) > 0:\n",
    "    print(f'Missing data for: {missing.index.tolist()}')\n",
    "else:\n",
    "    print('Full coverage — all 25 league-seasons have data')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ============================================================\n",
    "# SAVE RAW DATA\n",
    "# ============================================================\n",
    "output_path = DATA_PROC_DIR / 'standings_raw.csv'\n",
    "df_raw.to_csv(output_path, index=False)\n",
    "print(f'Saved to {output_path}')"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}